# Classical CNN — Matched-capacity ablation for the QCNN

Twin notebook of `hybrid_qcnn_multiseed_stats_v1.ipynb`. Architecture: same classical pipeline as the QCNN (channels 16 → 32 → 64), same 5% dropout, same depth, same training scheme. **The only change** with respect to the QCNN is that the final quanvolutional block is replaced by an equivalent classical `Conv2d(64, 64)`.

This is the honest ablation of the QCNN at equal representational capacity, distinct from the original 6/16 LeNet-5 baseline of Filippi (M.Sc. thesis, Pisa AY 2024/2025, supervisors Morsch + Cappuccio), which had substantially fewer parameters and was therefore structurally disadvantaged with respect to the Q-CNN in the comparison of Filippi's thesis.

The notebook executes $R=10$ runs with seeds `42 + run_idx * 111`, saves `Output_CCNN_v1_multiseed/results.json` in the extended format (per-item predictions included), and produces mean ± σ + Wilson 95% CI + bootstrap CI on the mean. The paired Wilcoxon QCNN-vs-CCNN comparison is done by the QCNN notebook (§18.7) or reproduced here in §19.5.

## §1 — Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import time, os, csv, copy, random, gc
from datetime import datetime
from dataclasses import dataclass, field
from typing import Optional, Literal, List, Dict
from pathlib import Path

import pytorch_lightning as L
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks import Callback, ModelCheckpoint, EarlyStopping
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchmetrics import Accuracy
from PIL import Image

import qiskit
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.primitives import StatevectorEstimator

# AerSimulator — fallback if not installed
try:
    from qiskit_aer import AerSimulator
    from qiskit_aer.primitives import EstimatorV2 as AerEstimator
    HAS_AER = True
except ImportError:
    HAS_AER = False
    print("⚠️  qiskit-aer not found — only StatevectorEstimator available")

QISKIT_VERSION = tuple(int(x) for x in qiskit.__version__.split('.')[:2])
assert QISKIT_VERSION >= (2, 0), f"Richiesto Qiskit >= 2.0, trovato {qiskit.__version__}"
print(f"Qiskit {qiskit.__version__} | PyTorch {torch.__version__} | Lightning {L.__version__}")
print(f"AerSimulator: {'✓' if HAS_AER else '✗'}")

## §2 — Configuration 

Architettura: Conv1(3→6,k5) → Pool → Conv2(6→6,k5) → Pool → Quanv(9q, 3×3) → Flatten → FC

In [ ]:
@dataclass
class CCNNConfig:
    """Matched-capacity CCNN configuration (thesis Ch.3, multi-seed campaign).

    Same pipeline as Filippi's HybridConvNet v2.0 (model C16-Q64)
    with the final quanvolutional replaced by a classical Conv2d(64,64).
    Same EuroSAT subset (2 classes), same batch_size, same learning
    rate, same schedule as Filippi v2.0.
    """

    # Dataset EuroSAT
    train_dir: str = "./dataset/training"
    val_dir: str = "./dataset/validation"
    img_size: int = 64
    in_channels: int = 3
    num_classes: int = 2                      # 2 classes (Forest vs AnnualCrop), consistent with Filippi §6.3.1
    selected_classes: Optional[List[str]] = None  # None = prime 2 disponibili
    max_samples_per_class: Optional[int] = 100   # 100 per classe (coerente con run effettivo CCNN, N_val=200)

    # Training
    batch_size: int = 16
    max_epochs: int = 10                      # 10 epoche (coerente con run effettivo CCNN R=10, vedi sec:stats-significance Cap.3)
    lr: float = 0.001
    weight_decay: float = 1e-4
    num_workers: int = 4
    early_stop_patience: int = 50             # Disabilitato di fatto

    # Dropout (matched a C16-Q64 di Filippi: 5%)
    dropout_rate: float = 0.05

    # Loop statistico
    num_stat_runs: int = 10                   # R=10 run multi-seed (Cap.3 Wave K)
    base_seed: int = 42

    # I/O
    run_name: str = "ccnn_matched_v1"
    output_dir: str = "Output_CCNN_v1_multiseed"
    seed: int = 42                            # Per costruzione train/val split (riproducibile)


# ═══════════════════════════════════════════
config = CCNNConfig()
L.seed_everything(config.seed)

print(f"Architettura CCNN matched-capacity:")
print(f"  Trunk:   Conv(3→16) → Pool → Conv(16→32) → Pool → Conv(32→64)")
print(f"  Replacement quanv:  Conv2d(64,64,kernel=3,padding=1)")
print(f"  Head:    Flatten → LazyLinear(5221) → Linear(5221, {config.num_classes})")
print(f"  Dropout: {config.dropout_rate} (matched a Filippi C16-Q64)")
print(f"  Classes: {config.num_classes}")
print(f"  Epoche:  {config.max_epochs}, batch {config.batch_size}, lr {config.lr}")
print(f"  Multi-seed: R={config.num_stat_runs}, base_seed={config.base_seed}")
print(f"  Output dir: {config.output_dir}")


## §3 — EuroSAT Dataset (class subset, consistent with Filippi)


In [ ]:
class EuroSATDataset(Dataset):
    """EuroSAT with class selection and sample-limit support."""

    def __init__(self, root_dir, transform=None, num_classes=4,
                 selected_classes=None, max_samples_per_class=None, seed=42):
        self.root_dir = root_dir
        self.transform = transform
        self.rng = random.Random(seed)

        available = sorted([d for d in os.listdir(root_dir)
                           if os.path.isdir(os.path.join(root_dir, d))])

        if selected_classes:
            self.classes = [c for c in selected_classes if c in available]
        else:
            self.classes = available[:num_classes]

        self.data = []
        for cls_idx, cls_name in enumerate(self.classes):
            cls_path = os.path.join(root_dir, cls_name)
            imgs = sorted([f for f in os.listdir(cls_path)
                          if f.lower().endswith(('.jpg', '.jpeg', '.png', '.tif', '.tiff'))])
            if max_samples_per_class and len(imgs) > max_samples_per_class:
                self.rng.shuffle(imgs)
                imgs = imgs[:max_samples_per_class]
            for img_file in imgs:
                self.data.append((os.path.join(cls_path, img_file), cls_idx))

        self.rng.shuffle(self.data)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


class EuroSATDataModule(L.LightningDataModule):
    MEAN = [0.485, 0.456, 0.406]
    STD  = [0.229, 0.224, 0.225]

    def __init__(self, config: CCNNConfig):
        super().__init__()
        self.config = config
        self.class_names = None

    def setup(self, stage=None):
        c = self.config
        train_tf = transforms.Compose([
            transforms.Resize((c.img_size, c.img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(self.MEAN, self.STD),
        ])
        val_tf = transforms.Compose([
            transforms.Resize((c.img_size, c.img_size)),
            transforms.ToTensor(),
            transforms.Normalize(self.MEAN, self.STD),
        ])

        if c.train_dir and os.path.exists(c.train_dir):
            self.train_dataset = EuroSATDataset(
                c.train_dir, train_tf, c.num_classes,
                c.selected_classes, c.max_samples_per_class, c.seed)
            self.class_names = self.train_dataset.classes
            print(f"  Training: {len(self.train_dataset)} img "
                  f"({len(self.class_names)} classes: {self.class_names})")
        else:
            print(f"  ⚠️  Train dir not found → synthetic dataset")
            self.train_dataset = self._synth(600)
            self.class_names = [f'C{i}' for i in range(c.num_classes)]

        if c.val_dir and os.path.exists(c.val_dir):
            self.val_dataset = EuroSATDataset(
                c.val_dir, val_tf, c.num_classes,
                c.selected_classes, c.max_samples_per_class, c.seed + 1)
            print(f"  Validation: {len(self.val_dataset)} img")
        else:
            print(f"  ⚠️  Val dir not found → synthetic dataset")
            self.val_dataset = self._synth(200)

    def _synth(self, n):
        class S(Dataset):
            def __init__(s, n, nc, sz, ch):
                s.data = [(torch.randn(ch, sz, sz), random.randint(0, nc-1)) for _ in range(n)]
            def __len__(s): return len(s.data)
            def __getitem__(s, i): return s.data[i]
        c = self.config
        return S(n, c.num_classes, c.img_size, c.in_channels)

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.config.batch_size,
                         shuffle=True, num_workers=self.config.num_workers,
                         pin_memory=True, persistent_workers=self.config.num_workers > 0)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.config.batch_size,
                         shuffle=False, num_workers=self.config.num_workers,
                         pin_memory=True, persistent_workers=self.config.num_workers > 0)

print("✓ EuroSATDataset + DataModule")

## §4 — Metrics Logger

In [ ]:
class MetricsLogger(Callback):
    """Records per-epoch metrics — logged to file + memory."""

    def __init__(self, log_dir=None):
        super().__init__()
        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []
        self.log_dir = log_dir
        self._csv_file = None
        self._csv_writer = None

    def on_fit_start(self, trainer, pl_module):
        if self.log_dir:
            os.makedirs(self.log_dir, exist_ok=True)
            self._csv_file = open(os.path.join(self.log_dir, 'metrics.csv'), 'w', newline='')
            self._csv_writer = csv.writer(self._csv_file)
            self._csv_writer.writerow(['epoch', 'train_loss', 'val_loss', 'train_acc', 'val_acc'])

    def on_validation_epoch_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        metrics = trainer.callback_metrics
        tl = metrics.get('train_loss_epoch', metrics.get('train_loss', torch.tensor(0))).item()
        vl = metrics.get('val_loss', torch.tensor(0)).item()
        ta = metrics.get('train_accuracy_epoch', metrics.get('train_accuracy', torch.tensor(0))).item()
        va = metrics.get('val_accuracy', torch.tensor(0)).item()

        self.train_losses.append(tl)
        self.val_losses.append(vl)
        self.train_accuracies.append(ta)
        self.val_accuracies.append(va)

        if self._csv_writer:
            self._csv_writer.writerow([trainer.current_epoch, f'{tl:.6f}', f'{vl:.6f}',
                                       f'{ta:.6f}', f'{va:.6f}'])
            self._csv_file.flush()

    def on_fit_end(self, trainer, pl_module):
        if self._csv_file:
            self._csv_file.close()

print("✓ MetricsLogger")

## §9 — Classical Conv Net (matched capacity)

Same pipeline as Filippi's Hybrid Q-CNN (model `C16-Q64`) with a single structural difference: the final quanvolutional block `QuantumConvLayer(in=64, out=64)` is replaced by `Conv2d(64, 64, kernel_size=3, padding=1)`. The representational capacity (parameter count, depth, receptive field) is preserved, so that the paired Wilcoxon QCNN-vs-CCNN is statistically honest.

In [ ]:
class ClassicalConvNet(nn.Module):
    """Matched-capacity ablation of the HybridConvNet of Filippi v2.0.

    Structure identical to the QCNN: three classical convolutional blocks at 16 / 32 / 64
    channels with interleaved MaxPool and Dropout. The difference is that the final
    quanvolutional block (QuantumConvLayer 64 → 64) is replaced by a
    Conv2d(64, 64, kernel=3, padding=1), preserving the number of
    parameters, the network depth, and the total receptive field.

    The fully-connected heads are identical to those of the QCNN, so that the
    number of trainable parameters of the classification head is
    rigorously the same.
    """

    def __init__(self, config):
        super().__init__()
        # in_channels = 3 per immagini RGB EuroSAT
        in_ch = 3

        # Classical convolutional trunk — identical to the QCNN
        self.conv1 = nn.Conv2d(in_ch, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Classical replacement of the QuantumConvLayer: same input/output shape
        # (64 → 64 canali, kernel 3, padding 1) per matched capacity
        self.conv4_classical = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(config.dropout_rate)
        self.relu = nn.ReLU(inplace=True)

        # Head identical to the QCNN (LazyLinear avoids hardcoding the flatten size)
        self.fc1 = nn.LazyLinear(5221)
        self.fc2 = nn.Linear(5221, config.num_classes)

    def forward(self, x):
        # Classical trunk
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.dropout(x)

        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = self.dropout(x)

        x = self.relu(self.conv3(x))
        x = self.dropout(x)

        # Replacement of the quanvolutional: equivalent classical Conv2d
        x = self.relu(self.conv4_classical(x))
        x = self.dropout(x)

        # Head
        x = torch.flatten(x, 1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x


## §10 — Lightning Classifier

In [ ]:
class ClassicalCNNClassifier(L.LightningModule):

    def __init__(self, model: ClassicalConvNet, config: CCNNConfig):
        super().__init__()
        self.model = model
        self.config = config
        self.loss_fn = nn.CrossEntropyLoss()
        self.train_acc = Accuracy(task="multiclass", num_classes=config.num_classes)
        self.val_acc = Accuracy(task="multiclass", num_classes=config.num_classes)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = logits.argmax(1)
        self.log('train_loss', loss, on_epoch=True, prog_bar=True)
        self.log('train_accuracy', self.train_acc(preds, y), on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.loss_fn(logits, y)
        preds = logits.argmax(1)
        self.log('val_loss', loss, on_epoch=True, prog_bar=True)
        self.log('val_accuracy', self.val_acc(preds, y), on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=self.config.lr,
                               weight_decay=self.config.weight_decay)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.config.max_epochs)
        return [opt], [sched]

print("✓ ClassicalCNNClassifier")

## §11 — Initialisation and test (CCNN)

Verifies that the `ClassicalConvNet` forward is stable and that the backward computes finite gradients.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Device: {device}')

# Smoke test of the classical model
torch.manual_seed(0)
model_test = ClassicalConvNet(config).to(device)
x_dummy = torch.randn(2, 3, 64, 64, device=device)
y_dummy = torch.randint(0, config.num_classes, (2,), device=device)
out = model_test(x_dummy)
print(f'Output shape: {tuple(out.shape)}')
loss = nn.functional.cross_entropy(out, y_dummy)
loss.backward()
print(f'Loss test: {loss.item():.4f}')

# Conteggio parametri (per confronto con QCNN)
n_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f'Parametri trainable: {n_params:,}')

del model_test, x_dummy, y_dummy, out, loss


## §12 — Dataset Loading

In [ ]:
data_module = EuroSATDataModule(config)
data_module.setup()

## §13 — Loop Statistico ($R$ run multi-seed)


In [ ]:
def create_fresh_model(config, device, seed, verbose=True):
    """New classical model with a different seed."""
    L.seed_everything(seed, workers=True)
    mdl = ClassicalConvNet(config)
    return mdl.to(device)


def collect_val_predictions(classifier, data_module, device):
    """Deterministic pass over the validation set AFTER trainer.fit, to
    raccogliere il vettore per-item di correttezza (lunghezza N_val).
    """
    classifier.eval()
    classifier = classifier.to(device)
    correct, labels = [], []
    with torch.no_grad():
        for x, y in data_module.val_dataloader():
            x, y = x.to(device), y.to(device)
            logits = classifier(x)
            preds = logits.argmax(dim=1)
            corr = (preds == y).to(torch.int64).cpu().tolist()
            labs = y.cpu().tolist()
            correct.extend(int(c) for c in corr)
            labels.extend(int(v) for v in labs)
    return correct, labels


def run_single_training(config, data_module, device, seed, run_idx, verbose=True):
    """A single training run (CCNN, no backend_manager)."""
    print(f"\n{'='*60}")
    print(f"  RUN {run_idx+1}/{config.num_stat_runs} — seed={seed}")
    print(f"{'='*60}")

    mdl = create_fresh_model(config, device, seed, verbose)
    classifier = ClassicalCNNClassifier(mdl, config)

    log_dir = os.path.join(config.output_dir, 'stat_runs', f'run_{run_idx:02d}_s{seed}')
    metrics_logger = MetricsLogger(log_dir=log_dir)

    callbacks = [
        metrics_logger,
        EarlyStopping(monitor='val_loss', patience=config.early_stop_patience,
                      mode='min', verbose=verbose),
    ]
    if run_idx == 0:
        best_ckpt = ModelCheckpoint(
            dirpath=log_dir, filename='best-{epoch}-{val_loss:.4f}',
            monitor='val_loss', save_top_k=1, mode='min')
        callbacks.append(best_ckpt)

    trainer = L.Trainer(
        max_epochs=config.max_epochs,
        callbacks=callbacks,
        logger=TensorBoardLogger(config.output_dir, name='stat_logs',
                                version=f'run_{run_idx:02d}'),
        accelerator='auto', devices=1,
        log_every_n_steps=1,
        enable_progress_bar=verbose,
        enable_checkpointing=(run_idx == 0),
    )

    t0 = time.time()
    trainer.fit(classifier, data_module)
    elapsed = time.time() - t0
    actual_epochs = trainer.current_epoch + 1

    val_correct, val_labels = collect_val_predictions(classifier, data_module, device)

    result = {
        'seed': seed, 'run_idx': run_idx, 'elapsed': elapsed,
        'actual_epochs': actual_epochs,
        'train_losses': list(metrics_logger.train_losses),
        'val_losses': list(metrics_logger.val_losses),
        'train_accuracies': list(metrics_logger.train_accuracies),
        'val_accuracies': list(metrics_logger.val_accuracies),
        'best_val_acc': max(metrics_logger.val_accuracies) if metrics_logger.val_accuracies else 0,
        'best_val_loss': min(metrics_logger.val_losses) if metrics_logger.val_losses else float('inf'),
        'final_train_acc': metrics_logger.train_accuracies[-1] if metrics_logger.train_accuracies else 0,
        'final_val_acc': metrics_logger.val_accuracies[-1] if metrics_logger.val_accuracies else 0,
        'val_correct_final': val_correct,
        'val_labels_final':  val_labels,
        'n_val': len(val_correct),
    }

    print(f"  Time: {elapsed:.0f}s ({actual_epochs} epoche)")
    print(f"  Best val_acc: {result['best_val_acc']:.4f}")
    print(f"  Final val_acc: {result['final_val_acc']:.4f} "
          f"({sum(val_correct)}/{len(val_correct)})")

    del classifier, trainer
    gc.collect()
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return result


In [ ]:
# ═══════════════════════════════════════════
# Loop statistico: R run con seed diversi
# Seeds: 42, 153, 264, 375, 486, 597, 708, 819, 930, 1041 (R=10)
# ═══════════════════════════════════════════

os.makedirs(config.output_dir, exist_ok=True)
results = []
seeds = [config.base_seed + run_idx * 111 for run_idx in range(config.num_stat_runs)]

for run_idx, seed in enumerate(seeds):
    result = run_single_training(
        config=config,
        data_module=data_module,
        device=device,
        seed=seed,
        run_idx=run_idx,
        verbose=True,
    )
    results.append(result)

print(f"\n{'='*60}")
print(f"  COMPLETATO: {len(results)} run salvati in memoria.")
print(f"  To persist to disk, run the saving cell in §19.")
print(f"{'='*60}")


## §14 — Tabella Riassuntiva

In [ ]:
def print_summary(results, config):
    print(f"\n{'─'*70}")
    print(f" {'Run':>4} │ {'Seed':>6} │ {'Epochs':>6} │ {'Best Val Acc':>12} │ "
          f"{'Best Val Loss':>13} │ {'Time':>8}")
    print(f"{'─'*70}")
    for r in results:
        print(f" {r['run_idx']+1:4d} │ {r['seed']:6d} │ {r['actual_epochs']:6d} │ "
              f"{r['best_val_acc']:12.4f} │ {r['best_val_loss']:13.4f} │ "
              f"{r['elapsed']:7.0f}s")
    print(f"{'─'*70}")
    accs = [r['best_val_acc'] for r in results]
    losses = [r['best_val_loss'] for r in results]
    print(f" Mean best val_acc:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f" Mean best val_loss: {np.mean(losses):.4f} ± {np.std(losses):.4f}")

print_summary(results, config)

## §15 — Visualisation: all curves

In [ ]:
def plot_all_curves(results, config):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'CCNN matched-capacity — {config.num_classes} classes, '
                 f'R={config.num_stat_runs} run', fontsize=13)

    colors = plt.cm.tab10(np.linspace(0, 1, len(results)))

    for i, r in enumerate(results):
        ep = range(1, len(r['train_losses'])+1)
        axes[0].plot(ep, r['train_losses'], '-', color=colors[i], alpha=0.6, label=f'Train R{i}')
        axes[0].plot(ep, r['val_losses'], '--', color=colors[i], alpha=0.8, label=f'Val R{i}')
        axes[1].plot(ep, r['train_accuracies'], '-', color=colors[i], alpha=0.6)
        axes[1].plot(ep, r['val_accuracies'], '--', color=colors[i], alpha=0.8)

    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss vs Epoch'); axes[0].legend(fontsize=7)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy vs Epoch')

    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'all_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()

plot_all_curves(results, config)

## §16 — Mean ± σ

In [ ]:
def plot_mean_bands(results, config):
    """Across-seed mean and dispersion of the train/val loss and accuracy curves.

    The +/- 1 sigma band is drawn as fill_between in the same
    colour as the line but with reduced alpha: matplotlib composes it
    automatically as a lighter tint of the line colour
    (explicit requirement for the thesis Ch.3 version).

    Palette colour-blind safe (RdBu/Greens vibrant): blu per Validation,
    arancione per Training, su entrambi i pannelli.
    """
    max_ep = max(len(r['val_losses']) for r in results)

    def pad(arr, length):
        padded = np.full(length, np.nan)
        padded[:len(arr)] = arr
        return padded

    train_losses = np.array([pad(r['train_losses'], max_ep) for r in results])
    val_losses   = np.array([pad(r['val_losses'],   max_ep) for r in results])
    train_accs   = np.array([pad(r['train_accuracies'], max_ep) for r in results])
    val_accs     = np.array([pad(r['val_accuracies'],   max_ep) for r in results])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'CCNN — mean \u00b1 \u03c3 (R = {len(results)} runs)',
                 fontsize=13)
    ep = np.arange(1, max_ep + 1)

    BAND_ALPHA = 0.22
    LINE_KWARGS = {'linewidth': 1.8, 'marker': 'o', 'markersize': 3}

    panels = [
        (train_losses, 'Train Loss',     '#e08214', axes[0]),
        (val_losses,   'Validation Loss', '#2166ac', axes[0]),
        (train_accs,   'Train Accuracy', '#e08214', axes[1]),
        (val_accs,     'Validation Accuracy', '#2166ac', axes[1]),
    ]
    for data, label, color, ax in panels:
        mean = np.nanmean(data, axis=0)
        std  = np.nanstd(data, axis=0)
        ax.plot(ep, mean, color=color, label=label, **LINE_KWARGS)
        ax.fill_between(ep, mean - std, mean + std,
                        color=color, alpha=BAND_ALPHA, linewidth=0)

    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss vs Epoch')
    axes[0].grid(True, linestyle='--', alpha=0.4)
    axes[0].legend(loc='upper right', fontsize=9)

    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy vs Epoch')
    axes[1].set_ylim(0.0, 1.02)
    axes[1].grid(True, linestyle='--', alpha=0.4)
    axes[1].legend(loc='lower right', fontsize=9)

    plt.tight_layout()
    fig_path = os.path.join(config.output_dir, 'mean_bands.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')

plot_mean_bands(results, config)


## §17 — Miglior run + confronto Filippi

In [ ]:
def plot_best_run(results, config):
    best = max(results, key=lambda r: r['best_val_acc'])
    ep = range(1, len(best['train_losses'])+1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'Best Run (seed={best["seed"]}) — '
                 f'Val Acc: {best["best_val_acc"]:.2%}', fontsize=13)

    # Loss
    axes[0].plot(ep, best['train_losses'], '-o', color='red', markersize=4, label='Train Loss')
    axes[0].plot(ep, best['val_losses'], '-o', color='green', markersize=4, label='Validation Loss')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss vs Epoch'); axes[0].legend()

    # Accuracy
    axes[1].plot(ep, best['train_accuracies'], '-o', color='red', markersize=4, label='Train Accuracy')
    axes[1].plot(ep, best['val_accuracies'], '-o', color='green', markersize=4, label='Validation Accuracy')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy vs Epoch'); axes[1].legend()

    # Highlight the jump after the first epoch (as reported in Filippi's thesis)
    if len(best['val_accuracies']) > 0:
        first_epoch_acc = best['val_accuracies'][0]
        axes[1].annotate(f'{first_epoch_acc:.1%}',
                        xy=(1, first_epoch_acc),
                        xytext=(3, first_epoch_acc - 0.1),
                        arrowprops=dict(arrowstyle='->', color='black'),
                        fontsize=10, fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'best_run.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # Confronto con Filippi
    print(f"\n📊 Confronto con Filippi:")
    print(f"  Filippi: val_acc ≈ 88% after epoch 1, loss ≈ 0.28")
    print(f"  Nostro:  val_acc = {best['best_val_acc']:.1%}, "
          f"loss = {best['best_val_loss']:.4f}")
    if len(best['val_accuracies']) > 0:
        print(f"  Jump epoca 1: {best['val_accuracies'][0]:.1%} "
              f"(Filippi: ~87%)")

plot_best_run(results, config)

## §18 — Distribution of the results

In [ ]:
def plot_distributions(results, config):
    if len(results) < 3:
        print("Servono almeno 3 run per le distribuzioni")
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    accs = [r['best_val_acc'] for r in results]
    losses = [r['best_val_loss'] for r in results]

    axes[0].hist(accs, bins=min(10, len(results)), edgecolor='black', alpha=0.7, color='steelblue')
    axes[0].axvline(np.mean(accs), color='red', linestyle='--', label=f'μ={np.mean(accs):.4f}')
    axes[0].set_xlabel('Best Val Accuracy'); axes[0].set_title('Distribuzione Accuracy')
    axes[0].legend()

    axes[1].hist(losses, bins=min(10, len(results)), edgecolor='black', alpha=0.7, color='salmon')
    axes[1].axvline(np.mean(losses), color='blue', linestyle='--', label=f'μ={np.mean(losses):.4f}')
    axes[1].set_xlabel('Best Val Loss'); axes[1].set_title('Distribuzione Loss')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'distributions.png'), dpi=150, bbox_inches='tight')
    plt.show()

plot_distributions(results, config)

## §18.5 — Statistica inferenziale (Wilson + bootstrap)

Addition of the inferential layer supporting the robustness claims of thesis Ch. 3:

* **Per-run Wilson 95% CI**: binomial interval on the single-run accuracy with $N_{\mathrm{val}}$ items. For the Filippi setup (4 classes, 100 img/class) $N_{\mathrm{val}}\approx 400$, so the CI half-width is of order $\pm 3$ percentage points at $\hat p \approx 0.9$ — informative.
* **Bootstrap 95% percentile CI** on the across-run mean of the *final* val accuracy (10\,000 resamples). Descriptive, complementary to $\bar m \pm \sigma$.
* **Plot of the Wilson intervals** for every run, with the across-run mean as reference.

In [ ]:
from scipy.stats import norm

def wilson_ci(k, n, alpha=0.05):
    """Wilson score interval for a binomial proportion k/n."""
    if n <= 0:
        return 0.0, 0.0
    z = float(norm.ppf(1.0 - alpha / 2.0))
    p = k / n
    denom = 1.0 + z * z / n
    centre = (p + z * z / (2.0 * n)) / denom
    half = z * np.sqrt(p * (1.0 - p) / n + z * z / (4.0 * n * n)) / denom
    return max(0.0, centre - half), min(1.0, centre + half)


def bootstrap_ci_mean(values, n_resamples=10000, alpha=0.05, seed=0):
    """Percentile bootstrap CI on the mean."""
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    n = values.size
    boots = np.empty(n_resamples)
    for i in range(n_resamples):
        boots[i] = values[rng.integers(0, n, size=n)].mean()
    lo, hi = np.percentile(boots, [100 * alpha / 2.0, 100 * (1 - alpha / 2.0)])
    return float(values.mean()), float(values.std(ddof=1)) if n > 1 else float('nan'), float(lo), float(hi)


def summary_statistical_table(results, config):
    """Prints a per-run table with Wilson 95% CIs and a summary row
    across runs with a bootstrap percentile CI on the mean."""
    n_val = results[0]['n_val'] if results else 0
    final_accs = np.array([r['final_val_acc'] for r in results], dtype=float)
    best_accs  = np.array([r['best_val_acc']  for r in results], dtype=float)

    print(f"\n{'\u2500' * 86}")
    print(f" Per-run final val accuracy + Wilson 95% CI (N_val = {n_val})")
    print(f"{'\u2500' * 86}")
    print(f" {'Run':>4} \u2502 {'Seed':>6} \u2502 {'k/n':>9} \u2502 "
          f"{'final acc':>10} \u2502 {'Wilson 95% CI':>22} \u2502 {'best acc':>10}")
    print(f"{'\u2500' * 86}")
    for r in results:
        k = int(round(r['final_val_acc'] * r['n_val']))
        lo, hi = wilson_ci(k, r['n_val'])
        ci_str = f"[{lo:.4f}, {hi:.4f}]"
        print(f" {r['run_idx']+1:>4d} \u2502 {r['seed']:>6d} \u2502 "
              f"{k:>4d}/{r['n_val']:<4d} \u2502 {r['final_val_acc']:>10.4f} \u2502 "
              f"{ci_str:>22} \u2502 {r['best_val_acc']:>10.4f}")
    print(f"{'\u2500' * 86}")

    m, s, lo, hi = bootstrap_ci_mean(final_accs, seed=1)
    print(f" Across-run final val acc: mean = {m:.4f}, std = {s:.4f}")
    print(f" Bootstrap 95% CI on the mean: [{lo:.4f}, {hi:.4f}]")

    m_b, s_b, lo_b, hi_b = bootstrap_ci_mean(best_accs, seed=2)
    print(f" Across-run BEST  val acc: mean = {m_b:.4f}, std = {s_b:.4f}")
    print(f" Bootstrap 95% CI on the mean: [{lo_b:.4f}, {hi_b:.4f}]")
    print(f"{'\u2500' * 86}")
    return {
        'final_mean': m, 'final_std': s, 'final_ci': (lo, hi),
        'best_mean': m_b, 'best_std': s_b, 'best_ci': (lo_b, hi_b),
        'n_val': n_val,
    }

stats_summary = summary_statistical_table(results, config)

In [ ]:
def plot_wilson_intervals(results, config):
    """Plots the per-run Wilson 95% CIs + across-run mean with bootstrap band."""
    n_val = results[0]['n_val']
    seeds = [r['seed'] for r in results]
    point  = np.array([r['final_val_acc'] for r in results], dtype=float)
    ks = (point * n_val).round().astype(int)
    lo_arr, hi_arr = [], []
    for k in ks:
        lo, hi = wilson_ci(int(k), n_val)
        lo_arr.append(lo); hi_arr.append(hi)
    lo_arr, hi_arr = np.array(lo_arr), np.array(hi_arr)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    x = np.arange(len(results))
    err_low = point - lo_arr
    err_high = hi_arr - point
    ax.errorbar(x, point, yerr=[err_low, err_high],
                fmt='o', color='#2166ac', ecolor='#2166ac',
                elinewidth=1.4, capsize=4, capthick=1.4,
                markersize=6, label='Single-run accuracy + Wilson 95% CI')

    mean = float(point.mean())
    _, _, b_lo, b_hi = bootstrap_ci_mean(point, seed=1)
    ax.axhline(mean, linestyle='--', color='#b2182b', linewidth=1.4,
               label=f'Across-run mean = {mean:.4f}')
    ax.axhspan(b_lo, b_hi, alpha=0.18, color='#b2182b',
               label=f'Bootstrap 95% CI on the mean: [{b_lo:.4f}, {b_hi:.4f}]')

    ax.set_xticks(x)
    ax.set_xticklabels([f's={s}' for s in seeds], rotation=30, fontsize=8)
    ax.set_ylabel('Validation accuracy')
    ax.set_title(f'Wilson 95% CI per run + bootstrap CI on the mean (N_val = {n_val})')
    ax.set_ylim(max(0.0, lo_arr.min() - 0.05), min(1.02, hi_arr.max() + 0.05))
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    fig_path = os.path.join(config.output_dir, 'wilson_intervals.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')

plot_wilson_intervals(results, config)

## §18.7 — Confronto cross-architecture (Wilcoxon paired)

CCNN vs QCNN comparison at equal **representational capacity** (16/32/64 channels, 5% dropout, same depth). The cells below load the `results.json` of the QCNN notebook (assumed to be in `../hybrid_qcnn/Output_QCNN_v1_multiseed/` — adapt the path to your local setup), compute the paired Wilcoxon signed-rank on the $R=10$ across-seed accuracy differences, and produce the cross-architecture plot.

Pairing is on `run_idx` (the seed scheme `base_seed + run_idx * 111` must be identical in the two notebooks).

**Note on Sebastianelli (2021)**: the purely-quantum reference model is cited in thesis Ch.3 only as a bibliographic reference; we do not include it in the paired Wilcoxon because the raw per-seed data are not available.

In [ ]:
# Path configuration: edit the two lines below to point to the
# results.json of the other architectures (classical CNN and purely-quantum
# reference). Non-existing paths are ignored.
OTHER_ARCH_PATHS = {
    'qcnn':   '../hybrid_qcnn/Output_QCNN_v1_multiseed/results.json',     # TO EDIT — path to the twin QCNN notebook
}

ARCH_LABELS = {
    'qcnn':   'Hybrid Q-CNN (this notebook)',
    'ccnn':   'Classical CNN (ablation)',
}

ARCH_COLORS = {
    'qcnn':   '#2166ac',
    'ccnn':   '#b2182b',
}

def _summarize_arch(arch_results, label):
    """Prints one diagnostic line for a loaded architecture."""
    R = len(arch_results)
    nv = arch_results[0]['n_val']
    final = np.array([r['final_val_acc'] for r in arch_results])
    print(f'  \u2713 {label:>10s}: R={R} run, N_val={nv}, '
          f'final_acc mean={final.mean():.4f}, std={final.std(ddof=1):.4f}')

# Always present: the results of this notebook (architecture 'qcnn')
arch_results = {'qcnn': results}
print('Architetture caricate:')
_summarize_arch(arch_results['qcnn'], 'qcnn')

import json as _json2
for arch, path in OTHER_ARCH_PATHS.items():
    if os.path.exists(path):
        try:
            with open(path) as f:
                payload = _json2.load(f)
            other_results = payload['results']
            # Check that it has the extended format (val_correct_final present)
            if 'val_correct_final' not in other_results[0]:
                print(f'  \u26a0\ufe0f  {arch}: results.json privo di val_correct_final, '
                      f'will be used for plotting only, not for the per-item Wilson CI.')
            arch_results[arch] = other_results
            _summarize_arch(other_results, arch)
        except Exception as e:
            print(f'  \u2717 {arch}: error while loading {path}: {e}')
    else:
        print(f'  \u2014 {arch}: path not found ({path}), skipping.')

print(f'\nTotal architectures available for the comparison: {len(arch_results)}')

In [ ]:
# Paired Wilcoxon signed-rank test on all pairs of available
# architectures. Pairing is on run_idx: for each i, we compare
# acc_A[i] with acc_B[i] (both produced by the same seed = base_seed + i*111).

from scipy.stats import wilcoxon
import itertools as _it

wilcoxon_results = {}

if len(arch_results) < 2:
    print('At least 2 architectures are needed for the paired Wilcoxon. '
          'Esegui i notebook gemelli e aggiorna OTHER_ARCH_PATHS.')
else:
    archs = list(arch_results.keys())
    print(f'\n{"\u2500"*100}')
    print(f' Wilcoxon signed-rank paired (final val accuracy across seeds)')
    print(f'{"\u2500"*100}')
    print(f' {"A vs B":>26s} | {"R":>3s} | {"mean(A)":>8s} | {"mean(B)":>8s} | '
          f'{"mean diff":>10s} | {"W stat":>8s} | {"p (two-sided)":>14s}')
    print(f'{"\u2500"*100}')

    for A, B in _it.combinations(archs, 2):
        accA = np.array([r['final_val_acc'] for r in arch_results[A]], dtype=float)
        accB = np.array([r['final_val_acc'] for r in arch_results[B]], dtype=float)
        R = min(len(accA), len(accB))
        if R < 2:
            print(f'  {A} vs {B}: R<2 after alignment, skipping.')
            continue
        accA, accB = accA[:R], accB[:R]
        diffs = accA - accB
        if np.all(diffs == 0):
            stat, pval = 0.0, 1.0
        else:
            try:
                res = wilcoxon(accA, accB, alternative='two-sided',
                               zero_method='wilcox', method='exact')
            except TypeError:
                res = wilcoxon(accA, accB, alternative='two-sided',
                               zero_method='wilcox')
            stat, pval = float(res.statistic), float(res.pvalue)
        label = f'{A} vs {B}'
        print(f' {label:>26s} | {R:>3d} | {accA.mean():>8.4f} | {accB.mean():>8.4f} | '
              f'{diffs.mean():>+10.4f} | {stat:>8.1f} | {pval:>14.4g}')
        wilcoxon_results[f'{A}_vs_{B}'] = {
            'n_pairs': int(R), 'mean_A': float(accA.mean()),
            'mean_B': float(accB.mean()), 'mean_diff': float(diffs.mean()),
            'W_statistic': stat, 'p_value_two_sided': pval,
            'differences_per_seed': diffs.tolist(),
        }
    print(f'{"\u2500"*100}')
    print('\nNota: con R=10 il p-value pi\u00f9 piccolo raggiungibile (esatto,\n'
          'two-sided) is 2/2^10 \u2248 0.002, attained when all 10\n'
          'signs of \u0394_i agree. A p~0.5 does NOT prove equivalence:\n'
          'only signals that the sign of the difference is not consistent.')

In [ ]:
def plot_cross_architecture_bands(arch_results, config):
    """Plots the three (or N) validation-accuracy-vs-epoch curves, with
    the across-seed \u00b11\u03c3 band drawn as a lighter tint of the line
    colour. The palette is colour-blind safe (RdBu + Greens).
    """
    if len(arch_results) < 2:
        print('Plot cross-arch saltato: serve almeno 2 architetture caricate.')
        return None

    # Lunghezza comune (padding NaN se differiscono)
    max_ep = max(len(r['val_accuracies']) for arch in arch_results.values() for r in arch)

    def pad(arr, length):
        padded = np.full(length, np.nan); padded[:len(arr)] = arr; return padded

    fig, ax = plt.subplots(figsize=(9, 5.2))
    ep = np.arange(1, max_ep + 1)
    BAND_ALPHA = 0.22
    LINE_KW = {'linewidth': 1.8, 'marker': 'o', 'markersize': 3}

    for arch, runs in arch_results.items():
        color = ARCH_COLORS.get(arch, None)
        label = ARCH_LABELS.get(arch, arch)
        val_accs = np.array([pad(r['val_accuracies'], max_ep) for r in runs])
        mean = np.nanmean(val_accs, axis=0)
        std = np.nanstd(val_accs, axis=0)
        ax.plot(ep, mean, color=color, label=f'{label} (R={len(runs)})', **LINE_KW)
        ax.fill_between(ep, mean - std, mean + std, color=color,
                        alpha=BAND_ALPHA, linewidth=0)

    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation accuracy')
    ax.set_title('Validation accuracy vs epoch \u2014 cross-architecture comparison')
    ax.set_ylim(0.0, 1.02)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    fig_path = os.path.join(config.output_dir, 'cross_architecture_with_std.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {fig_path}')
    return fig_path

plot_cross_architecture_bands(arch_results, config)

## §19 — Saving the results

In [ ]:
# Save the complete results
import json as _json
import csv as _csv

save_path = os.path.join(config.output_dir, 'results.json')
with open(save_path, 'w') as f:
    _json.dump({
        'architecture': 'ccnn_matched_capacity_v1',
        'config': {
            'num_classes': config.num_classes,
            'max_epochs': config.max_epochs,
            'batch_size': config.batch_size,
            'lr': config.lr,
            'weight_decay': config.weight_decay,
            'dropout_rate': config.dropout_rate,
            'max_samples_per_class': config.max_samples_per_class,
            'num_stat_runs': config.num_stat_runs,
            'base_seed': config.base_seed,
            'img_size': config.img_size,
            'in_channels': config.in_channels,
        },
        'results': results,
        'stats_summary': stats_summary if 'stats_summary' in dir() else {},
        'wilcoxon_results': wilcoxon_results if 'wilcoxon_results' in dir() else {},
    }, f, indent=2)
print(f"Results saved to: {save_path}")

# For each run, also write a predictions CSV with the per-item
# correctness vector, so that a downstream notebook can run the
# cross-architecture paired Wilcoxon by loading only the CSVs instead of the full JSON.
pred_dir = os.path.join(config.output_dir, 'predictions')
os.makedirs(pred_dir, exist_ok=True)
for r in results:
    csv_path = os.path.join(pred_dir, f"predictions_run{r['run_idx']:02d}_s{r['seed']}.csv")
    with open(csv_path, 'w', newline='') as f:
        w = _csv.writer(f)
        w.writerow(['item_idx', 'label', 'correct'])
        for i, (lab, corr) in enumerate(zip(r['val_labels_final'], r['val_correct_final'])):
            w.writerow([i, lab, corr])
print(f"Predictions per-run salvate in: {pred_dir}/")


## §19.5 — Rigenerazione plot da JSON salvati (post-hoc)

Optional section to **regenerate the plots from one or more previously saved `results.json`**, without re-running the training. Useful for:

* Changing the plot style (palette, band alpha, layout) and reviewing it on already-produced data.
* Generare il plot cross-architecture combinando run fatte in tempi diversi o su macchine diverse.
* Re-running the inferential statistics (Wilson, bootstrap, Wilcoxon) on an already-acquired dataset.
* Reproducing the numbers of a past run without spending compute time.

The cells are **self-contained**: they rebuild the `config` as a `SimpleNamespace` from the data saved in the JSON, and call the same plotting functions defined in the previous sections. They work both on `results.json` produced by this notebook and on those produced by the twin notebooks (classical CNN, purely-quantum), provided the format is the extended Ch.3 one (present from §19 onwards).

In [ ]:
# ─── Replay utilities ────────────────────────────────────────────
from types import SimpleNamespace
import json as _replay_json


def load_results_from_json(json_path):
    """Loads a results.json produced by §19 and returns (results, config).

    The returned config is a SimpleNamespace with all the fields saved in the
    JSON; output_dir = the directory containing the JSON is added, so that
    the regenerated figures are written next to the source file.
    Sensible defaults are set for the plot-title fields
    (num_conv_channels, num_qubits, num_classes) se assenti nel JSON.
    """
    if not os.path.exists(json_path):
        raise FileNotFoundError(f'JSON not found: {json_path}')
    with open(json_path) as f:
        payload = _replay_json.load(f)

    rs = payload.get('results', [])
    if not rs:
        raise ValueError(f'Nessun \"results\" nel JSON: {json_path}')

    cfg_dict = dict(payload.get('config', {}))
    cfg_dict.setdefault('output_dir', os.path.dirname(os.path.abspath(json_path)))
    cfg_dict.setdefault('num_conv_channels', 6)
    cfg_dict.setdefault('num_qubits', 9)
    cfg_dict.setdefault('num_classes', 4)

    cfg = SimpleNamespace(**cfg_dict)
    print(f'Caricati {len(rs)} run da {json_path}')
    print(f'  output_dir per le figure rigenerate: {cfg.output_dir}')
    return rs, cfg


def replay_single_arch_plots(json_path, run_summary=True, run_curves=False):
    """Regenerates all single-architecture plots from a saved results.json.
    Ritorna la tupla (results, config, stats_summary).

    run_curves=True also enables plot_all_curves (heavy if R > 10).
    """
    rs_r, cfg_r = load_results_from_json(json_path)

    if run_summary:
        print('\n--- print_summary ---')
        print_summary(rs_r, cfg_r)
    if run_curves:
        print('\n--- plot_all_curves ---')
        plot_all_curves(rs_r, cfg_r)
    print('\n--- plot_mean_bands ---')
    plot_mean_bands(rs_r, cfg_r)
    print('\n--- plot_best_run ---')
    plot_best_run(rs_r, cfg_r)
    print('\n--- plot_distributions ---')
    plot_distributions(rs_r, cfg_r)
    print('\n--- summary_statistical_table ---')
    stats_r = summary_statistical_table(rs_r, cfg_r)
    print('\n--- plot_wilson_intervals ---')
    plot_wilson_intervals(rs_r, cfg_r)
    return rs_r, cfg_r, stats_r

print('Utilities di replay caricate: load_results_from_json, replay_single_arch_plots')

In [ ]:
# ─── Example: single-architecture replay ─────────────────────────
# Decommenta e adatta il path al tuo results.json salvato.

# results_replay, config_replay, stats_replay = replay_single_arch_plots(
#     './Output_QCNN_v1_multiseed/results.json'
# )

In [ ]:
# ─── Replay cross-architecture (Wilcoxon paired + plot combinato) ─

def replay_cross_arch_plots(json_paths, output_dir=None, verbose=True):
    """Loads multiple results.json files and produces:
      (a) tabella Wilcoxon paired per ciascuna coppia di architetture,
      (b) figura cross_architecture_with_std.png (validation accuracy +
          banda \u00b11\u03c3 per ciascuna architettura),
      (c) the same wilcoxon_results as the live run, returned as output.

    Parametri:
      json_paths   dict {arch_name: path/results.json}
                   arch_name \u2208 chiavi di ARCH_LABELS ('qcnn', 'ccnn', 'pure_q')
      output_dir   directory dove scrivere la figura cross-arch.
                   If None, uses the directory of the first JSON found.
    """
    arch_results_r = {}
    if verbose:
        print('Loading architectures for the replay:')
    for arch, path in json_paths.items():
        if not os.path.exists(path):
            if verbose:
                print(f'  \u2014 {arch}: path not found ({path}), skipping.')
            continue
        try:
            with open(path) as f:
                payload = _replay_json.load(f)
            arch_results_r[arch] = payload['results']
            if verbose:
                R = len(arch_results_r[arch])
                nv = arch_results_r[arch][0].get('n_val', '?')
                final = np.array([r['final_val_acc'] for r in arch_results_r[arch]])
                print(f'  \u2713 {arch:>10s}: R={R}, N_val={nv}, '
                      f'mean={final.mean():.4f}, std={final.std(ddof=1):.4f}')
        except Exception as e:
            print(f'  \u2717 {arch}: error while loading: {e}')

    if len(arch_results_r) < 2:
        print('\nAt least 2 architectures are needed for the cross-arch replay. '
              'Esegui gli altri notebook e/o controlla i path.')
        return arch_results_r, {}

    # ── Wilcoxon paired ──
    from scipy.stats import wilcoxon as _wilc
    import itertools as _it_replay
    wlx = {}
    print(f'\n{"\u2500" * 100}')
    print(f' Wilcoxon signed-rank paired (replay)')
    print(f'{"\u2500" * 100}')
    print(f' {"A vs B":>26s} | {"R":>3s} | {"mean(A)":>8s} | {"mean(B)":>8s} | '
          f'{"mean diff":>10s} | {"W stat":>8s} | {"p (two-sided)":>14s}')
    print(f'{"\u2500" * 100}')
    for A, B in _it_replay.combinations(list(arch_results_r.keys()), 2):
        accA = np.array([r['final_val_acc'] for r in arch_results_r[A]], dtype=float)
        accB = np.array([r['final_val_acc'] for r in arch_results_r[B]], dtype=float)
        R = min(len(accA), len(accB))
        if R < 2:
            continue
        accA, accB = accA[:R], accB[:R]
        diffs = accA - accB
        if np.all(diffs == 0):
            stat, pval = 0.0, 1.0
        else:
            try:
                res = _wilc(accA, accB, alternative='two-sided',
                            zero_method='wilcox', method='exact')
            except TypeError:
                res = _wilc(accA, accB, alternative='two-sided',
                            zero_method='wilcox')
            stat, pval = float(res.statistic), float(res.pvalue)
        label = f'{A} vs {B}'
        print(f' {label:>26s} | {R:>3d} | {accA.mean():>8.4f} | {accB.mean():>8.4f} | '
              f'{diffs.mean():>+10.4f} | {stat:>8.1f} | {pval:>14.4g}')
        wlx[f'{A}_vs_{B}'] = {
            'n_pairs': int(R), 'mean_A': float(accA.mean()),
            'mean_B': float(accB.mean()), 'mean_diff': float(diffs.mean()),
            'W_statistic': stat, 'p_value_two_sided': pval,
            'differences_per_seed': diffs.tolist(),
        }
    print(f'{"\u2500" * 100}')

    # ── Plot cross-architecture ──
    if output_dir is None:
        # Use the directory of the first JSON as fallback
        first_path = next(p for p in json_paths.values() if os.path.exists(p))
        output_dir = os.path.dirname(os.path.abspath(first_path))
    os.makedirs(output_dir, exist_ok=True)
    cfg_replay = SimpleNamespace(
        output_dir=output_dir,
        num_qubits=9, num_conv_channels=6, num_classes=4,
    )
    plot_cross_architecture_bands(arch_results_r, cfg_replay)

    return arch_results_r, wlx


# ─── Usage example ───────────────────────────────────────────────
# Decommenta e adatta i path ai tuoi results.json:

# arch_replay, wilcoxon_replay = replay_cross_arch_plots(
#     json_paths={
#         'qcnn':   './Output_QCNN_v1_multiseed/results.json',
#         'ccnn':   './Output_CCNN_v1_multiseed/results.json',
#     },
#     output_dir='./Output_cross_arch_replay',
# )

## §20 — Cleanup

In [ ]:
# Cleanup memoria GPU/CPU (CCNN: niente backend quantum da chiudere)
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cleanup completato.')